# Pakai Dataset Siap Pakai (CSV / Kaggle)

Notebook ini menunjukkan cara load dataset sentimen yang sudah ada dalam format CSV.

**Apa itu dataset siap pakai?**
Dataset yang sudah di-download dan sudah ada label sentimennya. Biasanya dalam format CSV atau Excel. Kamu tinggal load dan pakai untuk training model.

**Sumber dataset populer:**
- [Kaggle Datasets](https://kaggle.com/datasets) - cari "indonesian sentiment"
- [GitHub repos](https://github.com) - banyak yang share dataset NLP Indonesia
- Scraping sendiri (dari notebook 01_youtube_dataset.ipynb)

In [ ]:
!pip install pandas matplotlib seaborn scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

## 1. Buat Dataset Contoh (Simulasi CSV)

Kita buat file CSV contoh dulu untuk simulasi. Di kasus nyata, kamu download CSV dari Kaggle atau sumber lain.

In [ ]:
# Buat dataset contoh dan simpan ke CSV
data = {
    'review': [
        'Produk bagus sekali kualitasnya memuaskan',
        'Pelayanan ramah dan cepat responsif',
        'Barang sesuai deskripsi mantap',
        'Kualitas premium harga murah recommended',
        'Pengiriman cepat packaging rapi sekali',
        'Seller sangat membantu dan sabar',
        'Sudah langganan selalu puas belanja di sini',
        'Produk original dan berkualitas tinggi',
        'Respon cepat barang sampai dengan baik',
        'Harga bersahabat kualitas tidak murahan',
        'Barang datang lebih cepat dari perkiraan suka',
        'Kualitas terbaik yang pernah saya coba',
        'Pelayanan bintang lima sangat memuaskan',
        'Produk jelek rusak baru dipakai sekali',
        'Pelayanan buruk tidak ada respon sama sekali',
        'Kecewa berat barang tidak sesuai gambar',
        'Penipuan uang sudah transfer barang tidak dikirim',
        'Kualitas sangat rendah jangan beli di sini',
        'Pengiriman sangat lama sudah sebulan belum sampai',
        'Barang cacat minta refund tidak ditanggapi',
        'Seller tidak bertanggung jawab sama sekali',
        'Produk palsu mengaku original mengecewakan',
        'Harga mahal kualitas sampah menyesal beli',
        'Rugi besar belanja di toko ini parah',
        'Barang rusak packaging hancur kecewa',
        'Pelayanan terburuk yang pernah saya alami',
        'Tidak recommended buang uang saja',
        'Produk biasa saja tidak ada yang istimewa',
        'Standar saja seperti toko online lainnya',
        'Lumayan oke untuk harga segitu',
        'Pengiriman normal tidak terlalu cepat',
        'Barang sesuai pesanan cukup puas',
        'Kualitas standar harga pas',
        'Cukup baik tapi bukan yang terbaik',
        'Biasa saja tidak terlalu suka tidak terlalu benci',
        'Harga sedang kualitas menengah',
        'Tidak ada masalah tapi juga tidak spesial',
        'Produk oke tapi packaging kurang rapi',
        'Seller lumayan responsif pengiriman standar',
        'Barang sampai tepat waktu kondisi normal',
    ],
    'sentimen': [
        'positif', 'positif', 'positif', 'positif', 'positif',
        'positif', 'positif', 'positif', 'positif', 'positif',
        'positif', 'positif', 'positif',
        'negatif', 'negatif', 'negatif', 'negatif', 'negatif',
        'negatif', 'negatif', 'negatif', 'negatif', 'negatif',
        'negatif', 'negatif', 'negatif', 'negatif',
        'netral', 'netral', 'netral', 'netral', 'netral',
        'netral', 'netral', 'netral', 'netral', 'netral',
        'netral', 'netral', 'netral',
    ]
}

# Simpan ke CSV
df = pd.DataFrame(data)
df.to_csv('contoh_dataset_sentimen.csv', index=False)
print(f'CSV dibuat: contoh_dataset_sentimen.csv ({len(df)} baris)')

## 2. Load Dataset dari CSV

Ini langkah utama — load file CSV ke pandas DataFrame.

**Kalau kamu punya file CSV sendiri**, ganti nama filenya di bawah.

In [ ]:
# Load CSV
# Ganti 'contoh_dataset_sentimen.csv' dengan file CSV kamu
df = pd.read_csv('contoh_dataset_sentimen.csv')

print(f'Total data: {len(df)}')
print(f'Kolom: {df.columns.tolist()}')
print(f'\nDistribusi sentimen:')
print(df['sentimen'].value_counts())
print()
df.head(10)

## 3. Cek Kualitas Data

Sebelum training, selalu cek dulu kualitas data.

In [ ]:
# Cek data kosong
print('Data kosong per kolom:')
print(df.isnull().sum())
print()

# Hapus baris yang kosong (kalau ada)
df = df.dropna()
print(f'Data setelah hapus yang kosong: {len(df)}')

# Cek duplikat
duplikat = df.duplicated().sum()
print(f'Data duplikat: {duplikat}')
if duplikat > 0:
    df = df.drop_duplicates()
    print(f'Data setelah hapus duplikat: {len(df)}')

In [ ]:
# Visualisasi distribusi sentimen
plt.figure(figsize=(8, 5))
counts = df['sentimen'].value_counts()
colors = ['#2ecc71', '#95a5a6', '#e74c3c']
sns.barplot(x=counts.index, y=counts.values, palette=colors)
plt.title('Distribusi Sentimen')
plt.xlabel('Sentimen')
plt.ylabel('Jumlah')
plt.tight_layout()
plt.show()

# Cek panjang teks
df['panjang_teks'] = df['review'].str.len()
print(f'\nStatistik panjang teks:')
print(df['panjang_teks'].describe())

## 4. Convert Label ke Angka

Model ML butuh angka, bukan teks. Convert label sentimen ke angka.

In [ ]:
# Mapping label teks ke angka
label_map = {'negatif': -1, 'netral': 0, 'positif': 1}
label_map_reverse = {-1: 'negatif', 0: 'netral', 1: 'positif'}

df['label'] = df['sentimen'].map(label_map)

print('Setelah convert:')
print(df[['review', 'sentimen', 'label']].head(10))

## 5. Preprocessing dan Training

In [ ]:
# Preprocessing
STOPWORDS = set([
    'yang', 'dan', 'di', 'ke', 'dari', 'untuk', 'dengan', 'pada',
    'ini', 'itu', 'tidak', 'juga', 'sudah', 'akan', 'saya', 'kami',
    'kita', 'mereka', 'dia', 'ada', 'sangat', 'saja', 'lagi', 'lah',
    'nya', 'tapi', 'kalau', 'oleh', 'saat', 'atau', 'hanya',
])

def bersihkan(teks):
    teks = teks.lower()
    teks = re.sub(r'[^a-z\s]', '', teks)
    teks = re.sub(r'\s+', ' ', teks).strip()
    kata_kata = teks.split()
    kata_kata = [k for k in kata_kata if k not in STOPWORDS]
    return ' '.join(kata_kata)

df['review_bersih'] = df['review'].apply(bersihkan)

# TF-IDF
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['review_bersih'])
y = df['label']

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training: {X_train.shape[0]}, Testing: {X_test.shape[0]}')

# Training
model = MultinomialNB()
model.fit(X_train, y_train)

# Evaluasi
y_pred = model.predict(X_test)
print(f'\nAkurasi: {accuracy_score(y_test, y_pred):.2%}')
print(f'\nClassification Report:')
print(classification_report(
    y_test, y_pred,
    target_names=['negatif', 'netral', 'positif'],
    zero_division=0
))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=[-1, 0, 1])

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negatif', 'Netral', 'Positif'],
            yticklabels=['Negatif', 'Netral', 'Positif'])
plt.title('Confusion Matrix')
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.tight_layout()
plt.show()

## 6. Uji dengan Review Baru

In [ ]:
def prediksi(teks):
    teks_bersih = bersihkan(teks)
    X = tfidf.transform([teks_bersih])
    label = model.predict(X)[0]
    proba = model.predict_proba(X)[0]
    sentimen = label_map_reverse[label]
    print(f'"{teks}"')
    print(f'  => Sentimen: {sentimen.upper()}')
    print(f'  => Confidence: {max(proba):.2%}')
    print()

prediksi('Barangnya bagus banget saya puas')
prediksi('Kecewa parah kualitasnya jelek')
prediksi('Biasa aja sih tidak terlalu spesial')
prediksi('Mantap jiwa seller terbaik pokoknya')
prediksi('Barang rusak tidak layak jual')

## 7. Load Dataset dari Kaggle

Kalau mau pakai dataset dari Kaggle, ada 2 cara:

### Cara A: Download manual
1. Buka kaggle.com, cari dataset "indonesian sentiment"
2. Download file CSV-nya
3. Upload ke Colab atau taruh di folder yang sama dengan notebook
4. Load dengan `pd.read_csv('nama_file.csv')`

### Cara B: Pakai Kaggle API (otomatis)

In [ ]:
# Cara B: Kaggle API
# 1. Install kaggle
!pip install kaggle --quiet

# 2. Upload kaggle.json ke Colab
#    (download dari kaggle.com > Settings > API > Create New Token)

import os
from google.colab import files

# Upload kaggle.json (file akan ter-upload dari komputer kamu)
uploaded = files.upload()

# Pindahkan ke folder yang benar
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 3. Download dataset
# Ganti dengan dataset yang kamu mau
# !kaggle datasets download -d username/nama-dataset

print('Upload kaggle.json dulu, lalu uncomment perintah download di atas')

## 8. Load Dataset dari URL (Google Drive, GitHub, dll)

Kalau dataset ada di internet, bisa langsung load dari URL.

In [ ]:
# Load CSV langsung dari URL
# Ganti URL dengan link CSV kamu

# Contoh dari GitHub (raw URL):
# url = 'https://raw.githubusercontent.com/user/repo/main/dataset.csv'
# df = pd.read_csv(url)

# Contoh dari Google Drive:
# 1. Share file sebagai "Anyone with the link"
# 2. Copy file ID dari URL: drive.google.com/file/d/FILE_ID/view
# url = f'https://drive.google.com/uc?id={FILE_ID}'
# df = pd.read_csv(url)

print('Ganti URL di atas dengan link CSV kamu, lalu jalankan cell ini')

## Ringkasan: Cara Load Dataset

| Sumber | Cara Load |
|---|---|
| File CSV lokal | `pd.read_csv('file.csv')` |
| Kaggle | Download manual atau pakai Kaggle API |
| Google Drive | `pd.read_csv('https://drive.google.com/uc?id=FILE_ID')` |
| GitHub | `pd.read_csv('https://raw.githubusercontent.com/...')` |
| Hugging Face | `load_dataset('nama_dataset')` (lihat notebook 02) |
| YouTube | YouTube API (lihat notebook 01) |

**Format CSV yang diharapkan:**
```
review,sentimen
"produk bagus",positif
"kualitas jelek",negatif
"biasa saja",netral
```

Yang penting ada **kolom teks** dan **kolom label sentimen**.